# QA generation with Qwen2.5-7B-Instruct

Turns the cleaned article paragraphs into question/answer pairs, on a free Kaggle GPU.
A 7B teacher generating training data for the 3B student we fine-tune on Day 5 - ordinary
knowledge distillation, and it means the whole dataset is reproducible by anyone who clones
the repo, with no API key and no spend.

## Why plain `transformers` and not vLLM

vLLM would be ~2x faster, but on a Tesla T4 it is a trap:

* Recent vLLM (0.15+) **crashes on T4** - CUTLASS DSL fails to detect compute capability
  sm_75. Last known-good is 0.8.x.
* Every vLLM version old enough to work on T4 pins an old torch (0.8.x -> torch 2.6,
  0.6.x -> torch 2.4). Installing one **downgrades Kaggle's torch and numpy**, which breaks
  the preinstalled torchaudio/jax/opencv and produces the
  `numpy.dtype size changed` binary-incompatibility error.

This notebook therefore uses the stock Kaggle image as-is: **no pip installs, no downgrades**.
The cost is speed (~2-3 h instead of ~1-1.5 h), which the resume cell makes safe.

## Before running

1. Create a Kaggle Dataset containing:
   - `paragraphs.jsonl`  - from `python -m src.dataset.clean`
   - `qa_prompt.py`      - copied from `src/dataset/`
   - `qa_validate.py`    - copied from `src/dataset/`

   The two modules are uploaded rather than pasted inline so the prompt and the validator
   stay single-sourced in git.
2. Attach it, then set **Accelerator -> GPU T4 x2** and **Internet -> On** (needed only to
   download the model weights from Hugging Face).
3. Run all. If the session dies, just run all again - it resumes.
4. Download `generations.jsonl` into `data/processed/`, then run `python -m src.dataset.build`.

In [ ]:
# No installs. Confirm the stock image is intact - if torch or numpy look downgraded,
# a previous pip install polluted the session: use Run -> Factory reset, not just Restart.
import numpy, torch

print(f"torch  {torch.__version__}")
print(f"numpy  {numpy.__version__}")
print(f"CUDA   {torch.cuda.is_available()}, {torch.cuda.device_count()} GPU(s)")
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    free, total = torch.cuda.mem_get_info(i)
    print(f"  [{i}] {name}  {total / 1e9:.0f} GB")

In [ ]:
import json, sys, time
from pathlib import Path

# Kaggle mounts datasets under /kaggle/input/<slug>, but the path differs depending on how
# the dataset was added. Rather than hardcode it, find the directory holding paragraphs.jsonl.
matches = list(Path("/kaggle/input").rglob("paragraphs.jsonl"))
assert matches, "paragraphs.jsonl not found under /kaggle/input - is the dataset attached?"
DATASET_DIR = matches[0].parent
OUTPUT_PATH = Path("/kaggle/working/generations.jsonl")
MODEL = "Qwen/Qwen2.5-7B-Instruct"
print("dataset dir:", DATASET_DIR)

sys.path.insert(0, str(DATASET_DIR))
from qa_prompt import build_messages

paragraphs = [json.loads(line) for line in
              (DATASET_DIR / "paragraphs.jsonl").read_text(encoding="utf-8").splitlines()
              if line.strip()]
print(f"{len(paragraphs)} paragraphs, {len({p['ticker'] for p in paragraphs})} companies")

In [ ]:
# Resume support, across sessions as well as within one.
#
# A committed run ("Save & Run All") starts with an empty /kaggle/working, so progress from
# an earlier run has to arrive as an attached dataset. This looks in both places and copies
# anything found in /kaggle/input into the working file, so the output of this run is always
# the complete set, not just what this run happened to add.
def collect_previous():
    seen = {}
    for path in sorted(Path("/kaggle/input").rglob("generations.jsonl")):
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                row = json.loads(line)
                seen[row["para_id"]] = line
    return seen

previous = collect_previous()
if previous:
    existing = set()
    if OUTPUT_PATH.exists():
        for line in OUTPUT_PATH.read_text(encoding="utf-8").splitlines():
            if line.strip():
                existing.add(json.loads(line)["para_id"])
    with OUTPUT_PATH.open("a", encoding="utf-8") as out:
        for para_id, line in previous.items():
            if para_id not in existing:
                out.write(line + "\n")
    print(f"carried {len(previous)} rows forward from attached dataset(s)")

done = set()
if OUTPUT_PATH.exists():
    for line in OUTPUT_PATH.read_text(encoding="utf-8").splitlines():
        if line.strip():
            done.add(json.loads(line)["para_id"])

pending = [p for p in paragraphs if p["para_id"] not in done]
print(f"{len(done)} already generated, {len(pending)} to go")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
# Decoder-only batch generation MUST pad on the left, or every sequence starts decoding
# from pad tokens and the output is garbage. This is the single easiest thing to get wrong.
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# float16, not bfloat16: T4 (Turing) has no bf16 support.
# device_map="auto" shards the 7B (~15 GB) across both T4s - it does not fit on one.
# sdpa attention works on Turing; flash-attention-2 requires Ampere or newer.
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)
model.eval()
print(model.hf_device_map)

In [ ]:
def render(paragraph):
    messages = build_messages(
        company=paragraph["company"],
        context=paragraph["text"],
        title=paragraph.get("title"),
    )
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print(render(pending[0])[-800:])

In [ ]:
BATCH = 16          # 7B fp16 across 2x T4; drop to 8 if you hit OOM
MAX_NEW_TOKENS = 300

torch.manual_seed(20260827)

# Recompute what is already done straight from disk, so re-running THIS cell after an
# interrupt always resumes - even if the resume cell above wasn't re-run first.
already = set()
if OUTPUT_PATH.exists():
    for line in OUTPUT_PATH.read_text(encoding="utf-8").splitlines():
        if line.strip():
            already.add(json.loads(line)["para_id"])

# Sorting by prompt length groups similar-length inputs into the same batch, so far less
# compute is wasted on padding. Worth roughly 20-30% of total runtime.
pending_sorted = sorted((p for p in paragraphs if p["para_id"] not in already),
                        key=lambda p: len(p["text"]))
print(f"resuming: {len(already)} done, {len(pending_sorted)} to go")
started = time.time()

with OUTPUT_PATH.open("a", encoding="utf-8") as out:
    for start in range(0, len(pending_sorted), BATCH):
        batch = pending_sorted[start:start + BATCH]
        inputs = tokenizer([render(p) for p in batch], return_tensors="pt",
                           padding=True, truncation=True, max_length=1600).to(model.device)

        with torch.inference_mode():
            generated = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=0.3,       # extraction, not creative writing - but not 0, so two
                top_p=0.9,             # questions from one paragraph aren't near-identical
                pad_token_id=tokenizer.pad_token_id,
            )

        # Slice off the prompt: keep only the newly generated tokens.
        completions = tokenizer.batch_decode(
            generated[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        for paragraph, completion in zip(batch, completions):
            out.write(json.dumps({"para_id": paragraph["para_id"],
                                  "response": completion}, ensure_ascii=False) + "\n")
        out.flush()

        finished = start + len(batch)
        rate = finished / (time.time() - started)
        print(f"{finished}/{len(pending_sorted)}  {rate:.1f}/s  "
              f"eta {(len(pending_sorted) - finished) / max(rate, 0.01) / 60:.0f} min", flush=True)

## Check the output before downloading

Runs the repo's real validator over a sample, so the rejection rate is known *here* rather
than after a download. A high rate concentrated in one reason means the prompt needs a fix,
not the filter.

**Run this after the first batch or two** - don't wait three hours to discover the prompt
is producing something unusable.

In [ ]:
from collections import Counter
from qa_validate import parse_response, validate

by_id = {p["para_id"]: p for p in paragraphs}
reasons, kept, sampled = Counter(), 0, 0

for line in OUTPUT_PATH.read_text(encoding="utf-8").splitlines()[:2000]:
    if not line.strip():
        continue
    row = json.loads(line)
    paragraph = by_id[row["para_id"]]
    pairs = parse_response(row["response"])
    if pairs is None:
        reasons["unparseable_json"] += 1
        continue
    first_word = paragraph["company"].lower().split()[0]
    for pair in pairs:
        sampled += 1
        reason = validate(pair, paragraph["text"], lambda q: first_word in q.lower())
        if reason:
            reasons[reason] += 1
        else:
            kept += 1

print(f"kept {kept} of {sampled} pairs ({100 * kept / max(sampled, 1):.1f}%)")
for reason, count in reasons.most_common():
    print(f"  {reason:22} {count}")

In [ ]:
for line in OUTPUT_PATH.read_text(encoding="utf-8").splitlines()[:5]:
    row = json.loads(line)
    print(by_id[row["para_id"]]["company"], "|", by_id[row["para_id"]]["text"][:110], "...")
    print(row["response"].strip()[:400])
    print("-" * 100)